# Lab 2 · Clean, connect and check

**In class:** Part A, guided, on a messy version of the penguins data · **At home:** Part B, the same steps on your own two tables
**Due:** Sunday, November 8 at 11:59 PM (extended for Election Day) · **Submit:** your repo URL on CourseWorks

Tonight's three skills:
1. **Clean** a messy CSV with pandas and load it into DuckDB
2. **Connect** two tables with a join, on a key you've checked
3. **Check** every join by counting rows before and after

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: messy penguins + a lookup table

## A1. Read the messy file with pandas

`penguins_raw.csv` is the same study as last week's penguins, in its original form: long column names, extra fields, comments. **pandas** is a Python library for working with tables; a table in pandas is called a **DataFrame**.

In [ ]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / "data" / "penguins_raw.csv")

print(raw.shape)   # (rows, columns)
raw.head()

How much is missing in each column? `isna()` marks missing values; `.sum()` counts them per column.

In [ ]:
raw.isna().sum()

Look at the species column. What would make it hard to use as a key?

In [ ]:
raw["Species"].value_counts()

**Notice:** long names with the Latin name in parentheses, and inconsistent capitals ("Adelie **P**enguin" vs. "Gentoo **p**enguin"). Text that *looks* the same to a person often won't match exactly in a join.

## A2. Three cleaning steps

**Step 1. Rename columns** to short, lowercase names with no spaces, so they're easy to type in SQL.

In [ ]:
penguins = raw.rename(columns={
    "studyName": "study",
    "Individual ID": "individual_id",
    "Species": "species_raw",
    "Island": "island",
    "Date Egg": "date_egg",
    "Clutch Completion": "clutch_completion",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
})
penguins.columns.tolist()

**Step 2. Standardize text.** Make `sex` lowercase, and pull the Latin name out of the parentheses into its own column. Scientific names are written the same way everywhere, which makes them a better key than the common name.

In [ ]:
penguins["sex"] = penguins["sex"].str.lower()
penguins["latin_name"] = penguins["species_raw"].str.extract(r"\((.*)\)")   # text inside ( )

penguins[["species_raw", "latin_name", "sex"]].drop_duplicates().head(6)

**Step 3. Fix types and keep what you need.** The egg date came in as text; make it a real date. Then keep only the columns we'll use.

In [ ]:
penguins["date_egg"] = pd.to_datetime(penguins["date_egg"])

penguins = penguins[[
    "study", "individual_id", "latin_name", "island", "date_egg", "clutch_completion",
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex",
]]
penguins.dtypes

## A3. Load it into DuckDB and verify with SQL

DuckDB can read a pandas DataFrame directly by its variable name. Always check the row count after moving data: **rows in should equal rows out.**

In [ ]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

print("rows in pandas:", len(penguins))
con.sql("SELECT count(*) AS rows_in_duckdb FROM penguins_clean")

In [ ]:
con.sql("DESCRIBE penguins_clean")

## A4. What is one row? Is the ID really an ID?

There's a column called `individual_id`. If it identified one penguin, every penguin would appear once per season. Let's check.

In [ ]:
con.sql("""
    SELECT count(*)                      AS rows,
           count(DISTINCT individual_id) AS distinct_ids
    FROM penguins_clean
""")

In [ ]:
con.sql("""
    SELECT study, latin_name, island, individual_id
    FROM penguins_clean
    WHERE individual_id = 'N11A1'
""")

**The same ID belongs to an Adélie penguin in one season and a Gentoo the next.** It can't be one bird. It's most likely a nest-and-adult label the researchers reused. So:

- `individual_id` **alone is not a key.** A column named "ID" isn't automatically unique; always check with `count(DISTINCT ...)`.
- **One row = one adult penguin at one nest in one breeding season.** That's the table's **grain**, and you should be able to say it for every table you use.

## A5. A lookup table to join

`species_lookup.csv` is a small reference table: one row per species, with its common name, Latin name, genus and group.

In [ ]:
con.sql(f"CREATE OR REPLACE TABLE species AS SELECT * FROM read_csv('{ROOT / 'data' / 'species_lookup.csv'}')")
con.sql("SELECT * FROM species")

Before joining, check that the key is **unique** in the lookup table. If a key appeared twice, every matching penguin would be copied twice.

In [ ]:
con.sql("""
    SELECT latin_name, count(*) AS n
    FROM species
    GROUP BY latin_name
    HAVING count(*) > 1
""")   # no rows back = the key is unique

## A6. Join, and count what comes out

`JOIN ... ON` lines up rows where the key matches. We expect **344** rows out, one per penguin row.

In [ ]:
con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")

**Only 276, not 344.** An inner `JOIN` silently drops rows that don't match. Nothing errored. This is a data cascade in the making.

## A7. Find what didn't match with a LEFT JOIN

A `LEFT JOIN` keeps every penguin row. Where there's no match, the lookup's columns come back empty (`NULL`), so we can find the problem rows.

In [ ]:
con.sql("""
    SELECT p.latin_name, count(*) AS unmatched_rows
    FROM penguins_clean p
    LEFT JOIN species s ON p.latin_name = s.latin_name
    WHERE s.latin_name IS NULL
    GROUP BY p.latin_name
""")

Compare with the lookup: the data says **Pygoscelis antarctic*a***, the lookup says **Pygoscelis antarctic*us***. Two sources spelled the same species differently, and 68 penguins vanished from the join.

## A8. Fix the key, then join again

Fix it at the source of the problem (the cleaning step), then reload and rejoin.

In [ ]:
penguins["latin_name"] = penguins["latin_name"].replace(
    {"Pygoscelis antarctica": "Pygoscelis antarcticus"}
)
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")   # now 344

Which way you join matters. Start from the **lookup** with a `LEFT JOIN` to see species that have no penguins:

In [ ]:
con.sql("""
    SELECT s.common_name, count(p.latin_name) AS penguin_rows
    FROM species s
    LEFT JOIN penguins_clean p ON p.latin_name = s.latin_name
    GROUP BY s.common_name
    ORDER BY penguin_rows DESC
""")

## A9. What a duplicate key does

Here we fake a lookup table that lists Gentoo twice, then join. Watch the row count.

In [ ]:
con.sql("""
    WITH species_with_dupe AS (
        SELECT * FROM species
        UNION ALL
        SELECT * FROM species WHERE common_name = 'Gentoo penguin'
    )
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species_with_dupe s ON p.latin_name = s.latin_name
""")   # 468: every Gentoo row was copied

**344 became 468**, and every total or average involving Gentoos would now be wrong. That's why you check key uniqueness (A5) before every join.

## A10. Use the join to answer a question

Now that the tables connect, ask a question that needs both: the lookup's readable names, the penguin table's measurements.

In [ ]:
con.sql("""
    SELECT s.common_name,
           p.island,
           count(*)                    AS penguins,
           round(avg(p.body_mass_g))   AS avg_mass_g
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
    GROUP BY s.common_name, p.island
    ORDER BY s.common_name, p.island
""")

### The join checklist (use it in Part B)

1. **Grain:** what is one row in each table?
2. **Key:** which column connects them, and is it **unique** in the lookup side?
3. **Clean the key** the same way on both sides (spelling, capitals, spaces, codes vs. names, types).
4. **Count rows** before and after. More rows = a duplicate key. Fewer = unmatched keys.
5. **LEFT JOIN … IS NULL** to see exactly what didn't match, then decide: fix, drop, or keep as unknown.

---
# Part B · At home: your own two tables

Apply the checklist to the two tables you chose in Lab 1. If you'd like to swap datasets, that's fine; you just need two public tables that share a column.

## B1. Load both tables with pandas

In [ ]:
FILE_A = "your_main_file.csv"     # <- your main table's file in data/
FILE_B = "your_second_file.csv"   # <- the related table's file in data/

a = pd.read_csv(ROOT / "data" / FILE_A)
b = pd.read_csv(ROOT / "data" / FILE_B)
print("A:", a.shape, "  B:", b.shape)
a.head()

In [ ]:
b.head()

**Grain** (double-click to edit). One row in table A is: ___. One row in table B is: ___.

## B2. Clean: at least three steps

Use the Part A steps as a model: rename columns, standardize text, fix types, drop columns you don't need. Write each step in its own line with a comment saying *why*.

In [ ]:
# Step 1: rename columns (short, lowercase, no spaces)
# a = a.rename(columns={"Country Code": "country_code", ...})

# Step 2: standardize the key text so both tables match
# a["country_code"] = a["country_code"].str.strip().str.upper()
# b["country_code"] = b["country_code"].str.strip().str.upper()

# Step 3: fix a type, or drop rows/columns you don't need
# a["year"] = a["year"].astype(int)

a.head()

## B3. Load into DuckDB and check the key

In [ ]:
con.sql("CREATE OR REPLACE TABLE table_a AS SELECT * FROM a")
con.sql("CREATE OR REPLACE TABLE table_b AS SELECT * FROM b")

KEY_A = "your_key_column_in_a"   # <- after your renames
KEY_B = "your_key_column_in_b"

# Is the key unique in table B? No rows back = unique.
con.sql(f"""
    SELECT "{KEY_B}", count(*) AS n
    FROM table_b
    GROUP BY "{KEY_B}"
    HAVING count(*) > 1
    LIMIT 10
""")

## B4. Join and count

In [ ]:
print("rows in table A:", con.sql("SELECT count(*) FROM table_a").fetchone()[0])

con.sql(f"""
    SELECT count(*) AS rows_after_join
    FROM table_a a
    JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
""")

## B5. Find what didn't match

In [ ]:
con.sql(f"""
    SELECT a."{KEY_A}", count(*) AS unmatched_rows
    FROM table_a a
    LEFT JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
    WHERE b."{KEY_B}" IS NULL
    GROUP BY a."{KEY_A}"
    ORDER BY unmatched_rows DESC
    LIMIT 20
""")

**What didn't match, and what will you do about it?** (fix the key, drop those rows, or keep them as unknown, and why)



## B6. Answer one question that needs both tables

Write a query that joins the tables and uses columns from **both**, e.g. average GDP by region, complaints per 1,000 residents by ZIP code.

In [ ]:
con.sql("""
    -- your query here: join table_a and table_b, then GROUP BY a column from table_b
    SELECT * FROM table_a LIMIT 5
""")

**What does the result tell you?** (one or two sentences)

## B7. Submit

1. Run the cell below.
2. Answer the two reflection questions in `labs/lab2.md`.
3. Save this notebook (Cmd+S), then commit and push: **Source Control** → `Lab 2` → **Commit** → **Sync Changes**.
4. Paste your repo URL into the Lab 2 assignment on CourseWorks.

In [ ]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")